<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/rivaasrasr-finetuning-conformer-ctc-nemo/nvidia_logo.png" style="width: 90px; float: right;">

# How to Fine-Tune a Riva ASR Acoustic Model with NVIDIA NeMo
This tutorial walks you through how to fine-tune an NVIDIA Riva nemotron-asr-streaming-rnnt-prompt acoustic model with NVIDIA NeMo.

## NVIDIA Riva NIM Overview

NVIDIA Riva ASR NIM APIs provide easy access to state-of-the-art automatic speech recognition (ASR) models for multiple languages. Riva ASR NIM models are built on the NVIDIA software platform, incorporating CUDA, TensorRT, and Triton to offer out-of-the-box GPU acceleration.

In this tutorial, we will interact with the automated speech recognition (ASR) APIs.

For more information about Riva ASR NIM, refer to the [Riva NIM documentation](https://docs.nvidia.com/nim/riva/asr/latest/overview.html).

## NeMo (Neural Modules)
[NVIDIA NeMo](https://developer.nvidia.com/nvidia-nemo) is an open-source framework for building, training, and fine-tuning GPU-accelerated speech AI and NLU models with a simple Python interface. For information about how to set up NeMo, refer to the [NeMo GitHub](https://github.com/NVIDIA/NeMo) instructions.

In [ ]:
"""
You can run either this tutorial locally (if you have all the dependencies and a GPU) or on Google Colab. If you run this tutorial in docker container with nemo supported you can skip this step.

Perform the following steps to setup in Google Colab:
1. Install cuda toolkit "conda install cudatoolkit"
2. Open a new Python 3 notebook. (3.8<=Python<=3.12)
3. Import this notebook from GitHub.
   a. Click **File** > **Upload Notebook** > **GITHUB** tab > copy/paste the GitHub URL.
4. Connect to an instance with a GPU.
   a. Click **Runtime** > Change the runtime type > select **GPU** for the hardware accelerator.
5. Run this cell to set up the dependencies.
6. Restart the runtime.
   a. Click **Runtime** > **Restart Runtime** for any upgraded packages to take effect.
"""

# Install Dependencies
!pip install wget
!apt-get install sox libsndfile1 ffmpeg libsox-fmt-mp3 jq
!pip install text-unidecode
!pip install matplotlib>=3.3.2
!pip install Cython
!pip3 install --no-cache-dir huggingface-hub==0.23.2

## Install NeMo
BRANCH = 'main'
!python -m pip install "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@{BRANCH}"

"""
Remember to restart the runtime for the kernel to pick up any upgraded packages (e.g. matplotlib)!
Alternatively, in the case where you want to use the "Run All Cells" (or similar) option,
uncomment `exit()` below to crash and restart the kernel.
"""
# exit()

---
## Fine-Tuning an ASR model with NeMo

### Download the Data

In this tutorial, we will use the popular AN4 dataset. Let's download it.

In [ ]:
! wget https://dldata-public.s3.us-east-2.amazonaws.com/an4_sphere.tar.gz  # for the original source, please visit http://www.speech.cs.cmu.edu/databases/an4/an4_sphere.tar.gz

After downloading, untar the dataset and move it to the correct directory.

In [ ]:
import os
DATA_DIR = os.getcwd()
os.environ["DATA_DIR"] = DATA_DIR
! tar -xvf an4_sphere.tar.gz
! mv an4 $DATA_DIR

### Pre-Processing

This step converts the `.mp3` files into `.wav` files and splits the data into training and testing sets. It also generates a "meta-data" file to be consumed by the data-loader for training and testing.

In [ ]:
import json, librosa, os, glob
import subprocess


source_data_dir = f"{DATA_DIR}/an4"
target_data_dir = f"{DATA_DIR}/an4_converted"

def an4_build_manifest(transcripts_path, manifest_path, target_wavs_dir):
    """Build an AN4 manifest from a given transcript file."""
    with open(transcripts_path, 'r') as fin:
        with open(manifest_path, 'w') as fout:
            for line in fin:
                # Lines look like this:
                # <s> transcript </s> (fileID)
                transcript = line[: line.find('(') - 1].lower()
                transcript = transcript.replace('<s>', '').replace('</s>', '')
                transcript = transcript.strip()

                file_id = line[line.find('(') + 1 : -2]  # e.g. "cen4-fash-b"
                audio_path = os.path.join(target_wavs_dir, file_id + '.wav')

                duration = librosa.core.get_duration(path=audio_path)

                # Write the metadata to the manifest
                metadata = {"audio_filepath": audio_path, "duration": duration, "text": transcript, "lang": "en-US", "target_lang": "en-US"}
                json.dump(metadata, fout)
                fout.write('\n')

"""Process AN4 dataset."""
if not os.path.exists(source_data_dir):
    link = 'http://www.speech.cs.cmu.edu/databases/an4/an4_sphere.tar.gz'
    raise ValueError(
        f"Data not found at `{source_data_dir}`. Please download the AN4 dataset from `{link}` "
        f"and extract it into the folder specified by the `source_data_dir` argument."
    )

# Convert SPH files to WAV files
sph_list = glob.glob(os.path.join(source_data_dir, '**/*.sph'), recursive=True)
target_wavs_dir = os.path.join(target_data_dir, 'wavs')
if not os.path.exists(target_wavs_dir):
    print(f"Creating directories for {target_wavs_dir}.")
    os.makedirs(os.path.join(target_data_dir, 'wavs'))

for sph_path in sph_list:
    wav_path = os.path.join(target_wavs_dir, os.path.splitext(os.path.basename(sph_path))[0] + '.wav')
    cmd = ["sox", sph_path, wav_path]
    subprocess.run(cmd, check=True)

# Build AN4 manifests
train_transcripts = os.path.join(source_data_dir, 'etc/an4_train.transcription')
train_manifest = os.path.join(target_data_dir, 'train_manifest.json')
an4_build_manifest(train_transcripts, train_manifest, target_wavs_dir)

test_transcripts = os.path.join(source_data_dir, 'etc/an4_test.transcription')
test_manifest = os.path.join(target_data_dir, 'test_manifest.json')
an4_build_manifest(test_transcripts, test_manifest, target_wavs_dir)


Let's listen to a sample audio file.

In [ ]:
# change path of the file here
import os
import IPython.display as ipd
path = os.environ["DATA_DIR"] + '/an4_converted/wavs/an268-mbmg-b.wav'
ipd.Audio(path)

### Tokenizers

If training dataset is not enough (<50 hrs) we recommend using tokenizers from pretrained model itself. The following tutorials explains how to use tokenizers from pretrained models for finetuning. If there's a change in vocab or you wish to train your own tokenizers you can use [NeMo tokenizer training script](https://github.com/NVIDIA/NeMo/blob/main/scripts/tokenizers/process_asr_text_tokenizer.py) and use [Hybrid model training script](https://github.com/NVIDIA/NeMo/blob/main/examples/asr/asr_hybrid_transducer_ctc/speech_to_text_hybrid_rnnt_ctc_bpe.py) to finetune the model on your data. Refer [asr-finetune-conformer-nemo](https://github.com/nvidia-riva/tutorials/blob/main/asr-finetune-conformer-ctc-nemo.ipynb) tutorial for more details.


#### Training nemotron-asr-streaming-rnnt-multilingual-prompt model

NeMo uses `.yaml` files to configure the training parameters. You may update them directly by editing the configuration file or from the command-line interface. For example, if the number of epochs needs to be modified, along with a change in the learning rate, you can add `trainer.max_epochs=100` and `optim.lr=0.02` and train the model.

The following sample command uses the `speech_to_text_finetune.py` script in the `examples` folder to train/fine-tune a nemotron-asr-streaming-rnnt-multilingual ASR model for 1 epoch. For other ASR models like Citrinet, Conformer, you may find the appropriate config files in the NeMo GitHub repo under [examples/asr/conf/](https://github.com/NVIDIA/NeMo/tree/main/examples/asr/conf).


In [ ]:
# To fully train the model, you'll need to increase trainer.max_epochs from 1.
# Empirical evidence suggests that around 200 epochs should suffice.
# If you are getting out of memory errors, reduce the ++model.train_ds.batch_duration value
NEMO_DIR = '/opt/prompt_model/NeMo'
BRANCH = 'main' # Subject to change to a fixed tag / version upon release
! git clone -b $BRANCH https://github.com/NVIDIA-NeMo/NeMo $NEMO_DIR

os.environ["PYTHONPATH"] = NEMO_DIR + ":" + os.environ.get("PYTHONPATH", "")

# HF_CKPT= <downloaded HF .nemo model path>

! python /{NEMO_DIR}/examples/asr/speech_to_text_finetune.py \
        --config-path="../asr/conf/fastconformer/cache_aware_streaming" --config-name=fastconformer_transducer_bpe_streaming_prompt.yaml \
        +init_from_nemo_model={HF_CKPT} \
        ++model.train_ds.manifest_filepath="$DATA_DIR/an4_converted/train_manifest.json" \
        ++model.validation_ds.manifest_filepath="$DATA_DIR/an4_converted/test_manifest.json" \
        ++model.optim.sched.d_model=1024 \
        ++trainer.devices=1 \
        ++trainer.max_epochs=1 \
        ++trainer.limit_train_batches=60 \
        ++trainer.precision=bf16 \
        ++model.train_ds.batch_duration=200 \
        ++model.optim.name="adamw" \
        ++model.optim.lr=0.1 \
        ++model.optim.weight_decay=0.001 \
        ++model.optim.sched.warmup_steps=100 \
        ++exp_manager.version=test \
        ++exp_manager.use_datetime_version=False \
        ++exp_manager.exp_dir=$DATA_DIR/checkpoints

### ASR Evaluation

Now that we have a model trained, we need to check how well it performs.

In [ ]:
nemo_file_path = os.path.join(os.environ["DATA_DIR"], "checkpoints/FastConformer-Transducer-BPE-Prompt-Streaming/test/checkpoints/FastConformer-Transducer-BPE-Prompt-Streaming.nemo")
! python /{NEMO_DIR}/examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py \
    model_path={nemo_file_path} \
    dataset_manifest="$DATA_DIR/an4_converted/test_manifest.json" \
    target_lang=auto \
    att_context_size="[56,3]" \
    decoder_type=rnnt \
    pad_and_drop_preencoded=true \
    batch_size=8 \
    cuda=0 \
    strip_lang_tags=false

## More Resources
You can find more information about working with NeMo's ASR models in the [ASR section](https://github.com/NVIDIA/NeMo/tree/main/tutorials/asr) of the NeMo tutorials.

## What's Next?

You can use NeMo to build custom models for your own applications, and deploy them with NVIDIA Riva! Refer to the [Riva NIM deployment tutorial](https://github.com/nvidia-riva/tutorials/blob/main/asr-deploy-am-and-ngram-lm.ipynb).